In [4]:
# Install dependencies
# %pip install python-dotenv typesafe_sdk
from dotenv import load_dotenv
import os
load_dotenv()
from typesafe_sdk import TypeSafeClient
MODEL_NAME="jev-latest"

In [ ]:
from dataclasses import dataclass

from typesafe_sdk import Choice, Noul, TypeSafeClient


@dataclass
class OrderIntent:
    side: str
    instrument: str
    size_fraction: str
    order_type: str
    rate: decimal | None = None


ALLOWED_INSTRUMENTS = {
    "EURUSD": "EUR/USD",
    "GBPUSD": "GBP/USD",
    "USDJPY": "USD/JPY",
    "EURGBP": "EUR/GBP",
}

SIZE_FRACTIONS = {
    "25_PERCENT": "One quarter of normal/default size",
    "50_PERCENT": "Half of normal/default size",
    "75_PERCENT": "Three quarters of normal/default size",
    "100_PERCENT": "Full normal/default size",
}


def parse_order_instruction(text: str) -> OrderIntent | None:

    with TypeSafeClient(
    api_key=os.environ["TYPESAFE_API_KEY"],
    base_url=os.environ["TYPESAFE_BASE_URL"]) as client:

        response = client.system_one(
            state={
                "instruction": text,
                "context": {
                    "allowed_instruments": list(ALLOWED_INSTRUMENTS.keys()),
                    "default_order_type": "MARKET",
                },
            },
            questions={

                # -------------------------
                # Typed arguments
                # -------------------------

                "side": Choice(
                    instructions=(
                        "Determine the trading side explicitly or clearly "
                        "implied by the trader instruction."
                    ),
                    criteria={
                        "BUY": "Trader wants to buy or go long",
                        "SELL": "Trader wants to sell or go short",
                    },
                ),

                "instrument": Choice(
                    instructions=(
                        "Which permitted FX instrument is the trader referring to?"
                    ),
                    criteria={
                        "EURUSD": "EUR/USD, euro-dollar, euro vs dollar",
                        "GBPUSD": "GBP/USD, cable, sterling-dollar",
                        "USDJPY": "USD/JPY, dollar-yen",
                        "EURGBP": "EUR/GBP, euro-sterling",
                    },
                ),

                "size_fraction": Choice(
                    instructions=(
                        "What fraction of the trader's normal/default order "
                        "size is requested?"
                    ),
                    criteria=SIZE_FRACTIONS,
                ),

                "order_type": Choice(
                    instructions="Which supported order type does the trader request?",
                    criteria={
                        "MARKET": "Execute at the current market price",
                        "LIMIT": "Execute only at a specified limit price",
                        "STOP": "Activate when a specified stop level is reached",
                    },
                ),
                "rate": Choice(
                    instructions=(
                        "If the trader specifies a rate, what is it? "
                        "Otherwise, return None."
                    ),
                    criteria={
                        "SPECIFIED": "Trader specifies a rate",
                        "NONE": "Trader does not specify a rate",
                    },
                ),

                # -------------------------
                # Ambiguity / safety checks
                # -------------------------

                "side_ambiguous": Noul(
                    instructions=(
                        "The instruction does NOT provide enough information "
                        "to determine BUY versus SELL reliably."
                    )
                ),

                "instrument_ambiguous": Noul(
                    instructions=(
                        "The instruction does NOT identify exactly one permitted "
                        "FX instrument reliably."
                    )
                ),

                "size_ambiguous": Noul(
                    instructions=(
                        "The requested trade size or size fraction is missing "
                        "or ambiguous."
                    )
                ),

                "order_type_ambiguous": Noul(
                    instructions=(
                        "The order type cannot be determined reliably from the "
                        "instruction and known defaults."
                    )
                ),
                "rate_ambiguous": Noul(
                    instructions=(
                        "The rate cannot be determined reliably from the "
                        "instruction and known defaults."
                    )
                ),
            },
        )

    side = response.choices["side"]
    instrument = response.choices["instrument"]
    size = response.choices["size_fraction"]
    order_type = response.choices["order_type"]
    rate = response.choices["rate"] if "rate" in response.choices else None

    # Noul gives probability that the proposition is TRUE.
    ambiguities = {
        "side": response.nouls["side_ambiguous"].noul,
        "instrument": response.nouls["instrument_ambiguous"].noul,
        "size": response.nouls["size_ambiguous"].noul,
        "order_type": response.nouls["order_type_ambiguous"].noul,
        "rate": response.nouls["rate_ambiguous"].noul
    }

    print("SIDE:", side.choice, side.confidence)
    print("INSTRUMENT:", instrument.choice, instrument.confidence)
    print("SIZE:", size.choice, size.confidence)
    print("ORDER TYPE:", order_type.choice, order_type.confidence)
    print("RATE:", rate.choice, rate.confidence, rate)

    print("AMBIGUITY:", ambiguities)

    # These thresholds are YOUR business rules, not Jev's.
    MIN_CHOICE_CONFIDENCE = 0.85
    MAX_AMBIGUITY = 0.20

    if any(p > MAX_AMBIGUITY for p in ambiguities.values()):
        return None

    if any(
        x.confidence < MIN_CHOICE_CONFIDENCE
        for x in [side, instrument, size, order_type, rate] if x
    ):
        return None

    return OrderIntent(
        side=side.choice,
        instrument=instrument.choice,
        size_fraction=size.choice,
        order_type=order_type.choice,
        rate= rate.choice if "rate" in response.choices else None
    )




In [17]:
instructions = [
    "Buy half my usual EUR/USD size at market.",
    "Buy £2m EURUSD at market, split 60% now and rest later.",
    "Sell all my GBP/USD at a limit of 1.3000.",
    "I want to go long on USD/JPY with 75% of my normal size.",
    "Please execute a stop order for EUR/GBP at 0.8500.",
]
if __name__ == "__main__":
    for instruction in instructions:
        print("*************")
        print(f"Instruction: {instruction}")
        order = parse_order_instruction(instruction)
        print(order)

*************
Instruction: Buy half my usual EUR/USD size at market.
SIDE: BUY 1.0
INSTRUMENT: EURUSD 1.0
SIZE: 50_PERCENT 1.0
ORDER TYPE: MARKET 1.0
RATE: NONE 1.0
AMBIGUITY: {'side': 0.31, 'instrument': 0.12, 'size': 0.66, 'order_type': 0.08, 'rate': 0.74}
None
*************
Instruction: Buy £2m EURUSD at market, split 60% now and rest later.
SIDE: BUY 1.0
INSTRUMENT: EURUSD 1.0
SIZE: 100_PERCENT 0.81
ORDER TYPE: MARKET 1.0
RATE: NONE 1.0
AMBIGUITY: {'side': 0.05, 'instrument': 0.07, 'size': 0.19, 'order_type': 0.09, 'rate': 0.79}
None
*************
Instruction: Sell all my GBP/USD at a limit of 1.3000.
SIDE: SELL 1.0
INSTRUMENT: GBPUSD 1.0
SIZE: 100_PERCENT 1.0
ORDER TYPE: LIMIT 1.0
RATE: SPECIFIED 1.0
AMBIGUITY: {'side': 0.03, 'instrument': 0.07, 'size': 0.89, 'order_type': 0.09, 'rate': 0.16}
None
*************
Instruction: I want to go long on USD/JPY with 75% of my normal size.
SIDE: BUY 1.0
INSTRUMENT: USDJPY 1.0
SIZE: 75_PERCENT 1.0
ORDER TYPE: MARKET 1.0
RATE: NONE 1.0
AMBIGU